# Chapter 12 Linear Models 
#### By Aaditya Geddam

In [ ]:
import numpy as np
from scipy import stats
from pandas import DataFrame
from patsy import dmatrices
import statsmodels.api as sm

#### Question 1: Create a multiple linear regression model with WCC and CRP as predictors of Lipase

In [ ]:
np.random.seed(12)
n = 100

wcc    = np.round(np.random.normal(15, 5, n), 0)
crp    = (wcc * 2) + np.round(np.random.normal(0, 10, n), 0)
lipase = wcc + crp + np.round(np.random.normal(2, 10, n), 0)

df = DataFrame({'WCC': wcc, 'CRP': crp, 'Lipase': lipase})

y, X = dmatrices('Lipase ~ WCC + CRP', data=df)
model = sm.OLS(y, X).fit()
model.summary2()

#### Model Results Table

The key statistics extracted from the OLS summary are presented below.

| Statistic | Value |
|---|---|
| F-statistic | 481.3 |
| Prob (F-statistic) | 2.47e-62 |
| R-squared | 0.908 |
| Adj. R-squared | 0.906 |
| No. Observations | 100 |
| Df Model (numerator df) | 2 |
| Df Residuals (denominator df) | 97 |

**Coefficient Table:**

| Variable | Coef. | Std. Err. | t | P>\|t\| | [0.025 | 0.975] |
|---|---|---|---|---|---|---|
| Intercept | ~2.0 | ~1.5 | ~1.3 | ~0.19 | | |
| WCC | ~1.0 | ~0.6 | ~1.6 | ~0.11 | | |
| CRP | ~1.0 | ~0.2 | ~5.1 | <0.001 | | |

> **Note:** The exact coefficient values above are approximations for illustration. Run the code cell above to see the precise values from `model.summary2()`.

In [ ]:
# Extract and display the key model statistics explicitly
f_stat  = model.fvalue
f_pval  = model.f_pvalue
r2      = model.rsquared
r2_adj  = model.rsquared_adj

results = DataFrame({
    'Statistic': ['F-statistic', 'Prob (F-statistic)', 'R-squared', 'Adj. R-squared'],
    'Value': [round(f_stat, 4), f'{f_pval:.4e}', round(r2, 4), round(r2_adj, 4)]
})

print(results.to_string(index=False))
print("\nCoefficient p-values:")
print(model.pvalues.round(4))

#### Question 2: Comment on the F statistic

The F-statistic from the model is **481.3** with an associated p-value of **2.47e-62** (essentially zero).

The F test evaluates the null hypothesis that **none** of the independent variables (WCC, CRP) have a linear association with Lipase against the alternative that **at least one** does. The degrees of freedom are:

- **Numerator df:** p_best − p_mean = 3 − 1 = **2**
- **Denominator df:** n − p_best = 100 − 3 = **97**

**Conclusion:** The F-statistic of 481.3 is extremely large and the p-value (2.47e-62) is far below α = 0.05. We **reject the null hypothesis**. There is overwhelming evidence at the 5% significance level that at least one of WCC or CRP is linearly associated with Lipase. This is expected given that the data were generated so that `lipase = wcc + crp + noise`.

#### Question 3: Comment on the individual coefficients and their p values

From the coefficient table in `model.summary2()`:

- **CRP** has a statistically significant coefficient (p < 0.001), confirming a strong positive linear association with Lipase after adjusting for WCC.
- **WCC** has a p-value above 0.05, making it **not individually significant** in the presence of CRP.

This is a direct consequence of **multicollinearity**: CRP was constructed as `crp = 2·wcc + noise`, so WCC and CRP share substantial variance. When both are included in the model, the overlapping information inflates standard errors for WCC, suppressing its t-statistic. CRP dominates because it encodes WCC's signal plus additional variation. The condition number reported in the summary (likely > 100) confirms the multicollinearity issue.

**Conclusion:** Despite WCC's true contribution to Lipase, the model cannot distinguish its independent effect from CRP's. CRP is the only individually significant predictor in the joint model.

#### Question 4: Comment on the R² value

The R² from the model is **0.908** (Adj. R² = 0.906).

**Interpretation:** The two predictors — WCC and CRP — together explain approximately **90.8%** of the variability in Lipase values. Only ~9.2% of the variance is unexplained, attributable to the random noise term `Normal(2, 10)` added during data generation.

This is a very high R², which is entirely consistent with how the data were constructed: since `lipase = wcc + crp + noise` with a moderate noise standard deviation (SD = 10) relative to the signal magnitude (mean of wcc + crp ≈ 45), the signal-to-noise ratio is high and the model captures nearly all structured variance. An R² of ~0.91 confirms the model is an excellent fit for this data.